In [1]:
cd ..

/Users/nguyenhuyhai/20232/Thesis/reputation-analysis


In [2]:
import numpy as np 
import pandas as pd 
import requests
import json
import pytz
from datetime import datetime
from multithread_processing.base_job import BaseJob
from utils.logger_utils import get_logger
from databases.mongodb_main import MongoDBMain



In [3]:
# with open('data/special_wallets/burn_wallets.json', 'r') as file:
#     burn_data = json.load(file)

# with open('data/special_wallets/hot_wallets.json', 'r') as file:
#     hot_data = json.load(file)

# burn_addresses = []    
# hot_addresses = []    

# for _, burn_value in burn_data.items():
#     burn_addresses.extend(burn_value)
# for _, hot_name in hot_data.items():
#     value_lst = list(hot_name.values())
#     addresses_lst = value_lst[1]
#     for _, hot_address in addresses_lst.items():
#         hot_addresses.extend(hot_address)
# burn_addresses = set(burn_addresses)
# burn_addresses.update(hot_addresses)
# hot_burn_addresses = set(burn_addresses)

In [4]:
class GetTransactionInfo(BaseJob):
    def __init__(self, chain, collections, historical_token_price, wallets_list: list, filename: str, start_date, end_date,
                 max_workers, batch_size):
        
        mongo = MongoDBMain(chain=chain)
        self.transac_collection = mongo._db[collections]        
        self.transaction_info = list()
        self.transfer_info = list()
        self.filename = filename
        self.start_date = start_date
        self.end_date = end_date
        self.historical_token_price = historical_token_price
        super().__init__(work_iterable=wallets_list,
                         max_workers=max_workers,
                         batch_size=batch_size)

    def _execute_batch(self, works):
        df_list = list()
        cursor =self.transac_collection.find({"block_timestamp" : {"$gte": self.start_date, "$lte": self.end_date},'from_address': {'$in': works}},{'from_address':1, 'to_address':1, 'value':1, 'block_timestamp':1,'transaction_type':1})
        for doc in cursor:
            single_transaction_info = dict(doc) 
            if single_transaction_info['value'] == '0':
                continue
            else:
                round_timestamp = (single_transaction_info['block_timestamp']  - single_transaction_info['block_timestamp']  % 86400) *1000
                # round_timestamp = (single_transaction_info['block_timestamp'] // 3600) * 3600000
                single_transaction_info['valueInUSD'] = int(single_transaction_info['value']) * self.historical_token_price['ethereum'][round_timestamp]
                df_list.append(single_transaction_info)    

        df_concat = pd.DataFrame(df_list)
        self.transaction_info.append(df_concat)
        
    def _end(self):
        super()._end()
        df = pd.concat(self.transaction_info, ignore_index=True)
        #return df
        df.to_csv(self.filename, index=False)


In [5]:
def format_prices(prices):
    formatted_prices = {}
    for price in prices:
        timestamp = price[0]
        price_value = price[1]
        formatted_prices[timestamp] = price_value
    return formatted_prices

def get_historical_prices(token_id, days=90, vs_currency='usd'):
    url = f'https://api.coingecko.com/api/v3/coins/{token_id}/market_chart?vs_currency={vs_currency}&days={days}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        prices = data.get('prices', [])
        formatted_prices = format_prices(prices)
        prices_dict = {token_id: formatted_prices}
        return prices_dict
    else:
        return None

In [6]:
logger = get_logger('User infromation exporter')
input_df = pd.read_json('data/reputation_score/aave_reputation.json')
addresses = input_df.wallet.unique().tolist()[0:100]

historical_token_price = get_historical_prices('ethereum')
historical_token_round_price = dict()
historical_token_round_price['ethereum'] = {(timestamp - (timestamp % 86400000)): value for timestamp, value in historical_token_price['ethereum'].items()}


In [7]:

job = GetTransactionInfo(chain='ethereum', 
                            collections='transactions', 
                            historical_token_price=historical_token_round_price,
                            wallets_list= addresses, 
                            filename='data/transaction_info/test.csv', 
                            start_date=1701388800, 
                            end_date= 1708992000, 
                            max_workers=4, 
                            batch_size=10)
job.run()


[02-28-2024 15:53:03 +07] [INFO] [BatchWorkExecutor] - Started work. Items to process: 100.


[02-28-2024 15:53:10 +07] [INFO] [BatchWorkExecutor] - 15 items processed. Progress is 15%.
[02-28-2024 15:53:16 +07] [INFO] [BatchWorkExecutor] - 30 items processed. Progress is 30%.
[02-28-2024 15:53:17 +07] [INFO] [BatchWorkExecutor] - 45 items processed. Progress is 45%.
[02-28-2024 15:53:18 +07] [INFO] [BatchWorkExecutor] - 60 items processed. Progress is 60%.
[02-28-2024 15:53:19 +07] [INFO] [BatchWorkExecutor] - 75 items processed. Progress is 75%.
[02-28-2024 15:53:22 +07] [INFO] [BatchWorkExecutor] - 90 items processed. Progress is 90%.
[02-28-2024 15:53:23 +07] [INFO] [BatchWorkExecutor] - 100 items processed. Progress is 100%.
[02-28-2024 15:53:23 +07] [INFO] [BatchWorkExecutor] - Finished work. Total items processed: 100. Took 0:00:20.554740.


## test

In [10]:
import pandas as pd
df = pd.read_csv("/Users/nguyenhuyhai/20232/Thesis/reputation-analysis/data/transaction_info/test.csv")
df

,_id,block_timestamp,from_address,to_address,value,valueInUSD
0,transaction_0x3f62c661c32fd307c0665c5d4fd84f3f...,1701833627,0x1c6b5795be43ddff8812b3e577ac752764635bc5,0xa397a8c2086c554b531c02e29f3291c9704b00c7,124000000000000000000,2.769401e+23
1,transaction_0x022094bdf7c74bfbdcc6404569a4a81e...,1701674135,0x1c6b5795be43ddff8812b3e577ac752764635bc5,0xa397a8c2086c554b531c02e29f3291c9704b00c7,96000000000000000000,2.134305e+23
2,transaction_0x6687ebcd2cf742a80d22f64ee4588657...,1708414907,0x3f08f17973ab4124c73200135e2b675ab2d263d9,0x4e0a22606a1abaa857ec4958ae9b0b38451dff4a,600000000000000000000,1.799647e+24
3,transaction_0x5aa5a8f4aa54438ec73d8a483a587660...,1708414247,0x3f08f17973ab4124c73200135e2b675ab2d263d9,0x4e0a22606a1abaa857ec4958ae9b0b38451dff4a,100000000000000000,2.999412e+20
4,transaction_0xc08b1cb012409962c50f87c688948c03...,1706168603,0x3f08f17973ab4124c73200135e2b675ab2d263d9,0x4e0a22606a1abaa857ec4958ae9b0b38451dff4a,100000000000000000,2.220750e+20
...,...,...,...,...,...,...
2710,transaction_0x7d2ba0a87809590b7264fe4c7cedad78...,1704864839,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xd322a49006fc828f9b5b37ab215f99b4e5cab19c,1255000000000000000000,3.200508e+24
2711,transaction_0x41f6b4ec31705cf1f379239c5b601769...,1704864455,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xd322a49006fc828f9b5b37ab215f99b4e5cab19c,1265000000000000000000,3.226010e+24
2712,transaction_0x93ead74e67574bb3fedd214563206d5e...,1704470099,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,1150000000000000000000,2.584940e+24
2713,transaction_0x6cb7df75fdc1d0752b38855728f78ddc...,1704469103,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,1705460000000000000000,3.833488e+24


In [16]:
input_df

,_id,NumberOfTokensBorrow,NumberOfTokensDeposit,NumberOfTokensHold,project,score,totalBorrowInUSD,totalDepositInUSD,totalHoldInUSD,wallet
0,aave_0x3744da57184575064838bbc87a0fc791f5e39ea2,0,0,1,aave,40468.188820,0.0,0.000000,2.108893e+07,0x3744da57184575064838bbc87a0fc791f5e39ea2
1,aave_0x5a801a9418d036fd453078c3adcb761fdc5ae695,0,0,1,aave,46432.913449,0.0,0.000000,2.723455e+07,0x5a801a9418d036fd453078c3adcb761fdc5ae695
2,aave_0x3555ef98046fac600c0b6529e7018ebeca176398,0,0,1,aave,34350.656068,0.0,0.000000,1.476900e+07,0x3555ef98046fac600c0b6529e7018ebeca176398
3,aave_0x61b0e6b68184eb0316b1285c4e76a15bfc7cd857,0,0,1,aave,34066.922075,0.0,0.000000,1.449354e+07,0x61b0e6b68184eb0316b1285c4e76a15bfc7cd857
4,aave_0x80845058350b8c3df5c3015d8a717d64b3bf9267,0,0,1,aave,27639.608332,0.0,0.000000,7.871301e+06,0x80845058350b8c3df5c3015d8a717d64b3bf9267
...,...,...,...,...,...,...,...,...,...,...
63181,aave_0x00000952c5165e391ed0f8adffcfaf45f3b80000,0,1,0,aave,20000.006045,0.0,1022.757331,0.000000e+00,0x00000952c5165e391ed0f8adffcfaf45f3b80000
63182,aave_0xbdede8c7555260b5af95b750e9e66761e04df9e4,0,1,0,aave,20000.000797,0.0,511.253697,0.000000e+00,0xbdede8c7555260b5af95b750e9e66761e04df9e4
63183,aave_0xa8b872aedb6843e241f0e9811940ccead032f909,0,1,0,aave,20000.000764,0.0,508.771391,0.000000e+00,0xa8b872aedb6843e241f0e9811940ccead032f909
63184,aave_0xceb4d74c140f0a7a35b7e37033ba53b8ea7ad327,0,1,0,aave,20000.000656,0.0,500.605429,0.000000e+00,0xceb4d74c140f0a7a35b7e37033ba53b8ea7ad327


In [19]:
info_wallet = df.groupby(['from_address', 'to_address']).agg({"block_timestamp":tuple, "valueInUSD": np.sum}).reset_index().rename(columns={"from_address": "wallet","to_address":"sending_wallet"})

info_wallet = info_wallet.merge(input_df[['wallet', 'NumberOfTokensBorrow',	'NumberOfTokensDeposit', 'NumberOfTokensHold', 'totalBorrowInUSD', 'totalDepositInUSD',	'totalHoldInUSD', 'score']], on='wallet', how='inner')
info_wallet

,wallet,sending_wallet,block_timestamp,valueInUSD,NumberOfTokensBorrow,NumberOfTokensDeposit,NumberOfTokensHold,totalBorrowInUSD,totalDepositInUSD,totalHoldInUSD,score
0,0x0f11640bf66e2d9352d9c41434a5c6e597c5e4c8,0x4dbd4fc535ac27206064b68ffcf827b0a60bab3f,"(1705944059,)",1.158836e+20,0,1,1,0.000000e+00,1.388998e+06,1.434273e+06,21501.018232
1,0x0f11640bf66e2d9352d9c41434a5c6e597c5e4c8,0x72ce9c846789fdb6fc1f34ac4ad25dd9ef7031ef,"(1705946567, 1705945715, 1705945583)",2.665356e+18,0,1,1,0.000000e+00,1.388998e+06,1.434273e+06,21501.018232
2,0x0f1dfef1a40557d279d0de6e49ab306891a638b8,0x036676389e48133b63a802f8635ad39e752d375d,"(1708955003,)",1.594608e+22,0,2,0,0.000000e+00,2.009030e+06,0.000000e+00,20157.134523
3,0x0f1dfef1a40557d279d0de6e49ab306891a638b8,0x109ec853789064ef391e003c4468687f9942c671,"(1707230075, 1707240239, 1706195543, 170301382...",3.618018e+26,0,2,0,0.000000e+00,2.009030e+06,0.000000e+00,20157.134523
4,0x0f1dfef1a40557d279d0de6e49ab306891a638b8,0x1111111254eeb25477b68fb85ed929f73a960582,"(1707135383, 1706871167, 1705534943, 170543785...",1.042931e+26,0,2,0,0.000000e+00,2.009030e+06,0.000000e+00,20157.134523
...,...,...,...,...,...,...,...,...,...,...,...
374,0xf368d43f148e1803ec793670183b0ca6a07d3898,0xd322a49006fc828f9b5b37ab215f99b4e5cab19c,"(1703956583, 1705967879, 1704029171, 170402906...",1.543674e+24,2,2,0,1.466319e+07,2.792593e+07,0.000000e+00,22958.190064
375,0xf368d43f148e1803ec793670183b0ca6a07d3898,0xeffc18fc3b7eb8e676dac549e0c693ad50d1ce31,"(1705967807, 1704029123)",3.342726e+23,2,2,0,1.466319e+07,2.792593e+07,0.000000e+00,22958.190064
376,0xf368d43f148e1803ec793670183b0ca6a07d3898,0xf047299c3836e033dde7687574e816f25f611be4,"(1707325571, 1705329623, 1706793479)",1.388083e+24,2,2,0,1.466319e+07,2.792593e+07,0.000000e+00,22958.190064
377,0xf368d43f148e1803ec793670183b0ca6a07d3898,0xfe655056aa04d10a9a433f02eb3ed3bf9ee9163b,"(1707401879, 1706793707, 1703880347)",1.761523e+24,2,2,0,1.466319e+07,2.792593e+07,0.000000e+00,22958.190064
